# Falcon 임베딩 생성
이 노트북은 Falcon만 생성합니다. 기본 cuda_8bit 설정은 NVIDIA GPU가 필요합니다. CPU/MPS는 research.yaml의 quantization=none 설정을 선택하세요. LLaMA는 모델 접근 권한과 HF 로그인/HF_TOKEN이 필요할 수 있습니다. 토큰을 셀에 쓰지 마세요.

In [ ]:
from pathlib import Path
from llm_bertopic.config import load_config, effective_config

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
print("Repository:", ROOT)


In [ ]:
# 전처리: full / minimal_keep_stopwords / minimal_remove_stopwords
CONFIG = ROOT / "configs/cuda_8bit.yaml"
MODEL = "falcon"
DATASETS = ["bbc"]  # 전체 실행: ["newsgroup20", "bbc", "imdb"]
PREPROCESSING = "full"
cfg = load_config(CONFIG, root=ROOT)
cfg["preprocessing"]["mode"] = PREPROCESSING
cfg["vectorizer"]["stop_words"] = None if PREPROCESSING == "minimal_keep_stopwords" else "english"
cfg["experiment"]["datasets"] = DATASETS
cfg["experiment"]["models"] = [MODEL]
# 필요 시 변경: cfg["embeddings"][MODEL]["batch_size"] = 2
cfg["embeddings"][MODEL]


## 문서 준비 및 임베딩 저장
문서 순서와 설정이 같은 캐시가 있으면 재사용합니다. 설정이나 문서가 바뀌면 별도 캐시를 생성합니다. legacy_mean은 패딩을 포함하므로 batch_size 변경도 별도 실험입니다.

In [ ]:
from llm_bertopic.data import prepare_dataset
from llm_bertopic.embeddings import ensure_embeddings, embedding_location

for dataset in DATASETS:
    corpus = prepare_dataset(cfg, dataset, MODEL)
    matrix = ensure_embeddings(cfg, dataset, MODEL, corpus)
    folder, identity = embedding_location(cfg, dataset, MODEL, corpus)
    print(dataset, MODEL, matrix.shape, folder)
    del matrix
